In [1]:
import pandas as pd
from ortools.sat.python import cp_model


pers_df = pd.read_excel('Phase2.xlsx', sheet_name='Pers')
pos_df = pd.read_excel('Phase2.xlsx', sheet_name='Pos')

In [2]:
# Mapping
citizenship_map = {"Local": 1, "PR": 2, "Foreigner": 3, "Any": 4}
field_map = {"IT":1, "MED":2, "SALES":3, "ENGINEER":4, "RANDOM":5, "LEAD":6, "ANY":7}
tag_map = {"HR":1, "Int":2, "Ops":3, "Logs":4, "Plans":5, "Train":6, "Main":7}
tier_map = {"Junior":1, "Mid":2, "Senior":3}

pers_df["Citizenship"] = pers_df["Citizenship"].map(citizenship_map)
pers_df["Tier"] = pers_df["Tier"].map(tier_map)
pers_df["Field"] = pers_df["Field"].map(field_map)

pos_df["Req_Citizenship"] = pos_df["Req_Citizenship"].map(citizenship_map)
pos_df["Req_Tier"] = pos_df["Req_Tier"].map(tier_map)
pos_df["Req_Field"] = pos_df["Req_Field"].map(field_map)
pos_df["Tag"] = pos_df["Tag"].map(tag_map)

# pos_df = pos_df.sample(n=1000).reset_index(drop=True)

In [3]:
model = cp_model.CpModel()
solver = cp_model.CpSolver()

no_of_people = len(pers_df)
no_of_desk = len(pos_df)

assignments = {}
penalties   = {}

In [4]:
for i in range(no_of_people):
    for j in range(no_of_desk):
        pers = pers_df.loc[i]
        pos  = pos_df.loc[j]

        citizen_match = (pers["Citizenship"] == pos["Req_Citizenship"]) or (pos["Req_Citizenship"] == 4)
        field_match   = (pers["Field"] == pos["Req_Field"]) or (pos["Req_Field"] == 7)

        if citizen_match and field_match:
            assignments[i,j] = model.NewBoolVar(f"x_{i}_{j}")
            pers_tier = pers["Tier"]
            pos_tier  = pos["Req_Tier"]

            penalty_var = model.NewIntVar(0, 10, f"penalty_{i}_{j}")
            penalties[i,j] = penalty_var
            
            if pers_tier == pos_tier:
                model.Add(penalty_var == 0).OnlyEnforceIf(assignments[i,j])
            elif (pers_tier == 1 and pos_tier == 2) or \
                 (pers_tier == 2 and pos_tier == 1) or \
                 (pers_tier == 2 and pos_tier == 3) or \
                 (pers_tier == 3 and pos_tier == 2):
                model.Add(penalty_var == 1).OnlyEnforceIf(assignments[i,j])

In [5]:
for i in range(no_of_people):
    feasible_jobs = [j for j in range(no_of_desk) if (i,j) in assignments]
    if feasible_jobs:
        model.Add(sum(assignments[i,j] for j in feasible_jobs) == 1)
    else:
        print(f"Warning: Person {pers_df.loc[i,'Name_ID']} has no feasible job!")
        
for j in range(no_of_desk): 
    feasible_people = [i for i in range(no_of_people) if (i,j) in assignments]
    if feasible_people:
        model.Add(sum(assignments[i,j] for i in feasible_people) <= 1)

In [8]:
appointment_mismatch = model.NewIntVar(0, no_of_people * no_of_desk * 10, "appointment_mismatch")
penalty_sum_list = [penalties[i,j] * 5 for i,j in penalties]
model.Add(appointment_mismatch == sum(penalty_sum_list))

model.Maximize(sum(assignments.values()) - appointment_mismatch)

In [9]:
status_code = solver.Solve(model)
print(f"{solver.StatusName(status_code)} ({status_code})")
print("Objective value:", solver.ObjectiveValue())
print("Best bound:", solver.BestObjectiveBound())

relaxed_limit = int(solver.ObjectiveValue() * 1.05)
model.Add(appointment_mismatch <= relaxed_limit)

OPTIMAL (4)
Objective value: 1997.0
Best bound: 1997.0


In [10]:
# Solve again
status_code = solver.Solve(model)
print(f"Relaxed solve: {solver.StatusName(status_code)} ({status_code})")
print("Objective value (relaxed):", solver.ObjectiveValue())

Relaxed solve: OPTIMAL (4)
Objective value (relaxed): 1997.0


In [11]:
inv_citizenship_map = {v: k for k, v in citizenship_map.items()}
inv_field_map = {v: k for k, v in field_map.items()}
inv_tag_map = {v: k for k, v in tag_map.items()}
inv_tier_map = {v: k for k, v in tier_map.items()}

pers_df["Citizenship"] = pers_df["Citizenship"].map(inv_citizenship_map)
pers_df["Tier"] = pers_df["Tier"].map(inv_tier_map)
pers_df["Field"] = pers_df["Field"].map(inv_field_map)

pos_df["Req_Citizenship"] = pos_df["Req_Citizenship"].map(inv_citizenship_map)
pos_df["Req_Tier"] = pos_df["Req_Tier"].map(inv_tier_map)
pos_df["Req_Field"] = pos_df["Req_Field"].map(inv_field_map)
pos_df["Tag"] = pos_df["Tag"].map(inv_tag_map)

In [12]:
print(pos_df.head(5))
print(pers_df.head(5))

   Desk_ID Req_Citizenship Req_Tier Req_Field    Tag  Level
0  D000001              PR   Junior     SALES     HR      1
1  D000002       Foreigner   Junior  ENGINEER  Train      1
2  D000003           Local      Mid       ANY    Int      2
3  D000004              PR   Junior     SALES   Logs      1
4  D000005           Local   Junior        IT   Main      1
   Name_ID           Name Citizenship    Tier Field  Pass_Test1  Pass_Test2
0  P000001  Roxuqy Tiwoni       Local     Mid    IT        True       False
1  P000002    Lyxu Subyzu   Foreigner  Junior  LEAD       False       False
2  P000003  Cagyla Gedyzo       Local     Mid   MED        True       False
3  P000004    Huwyle Wola          PR     Mid  LEAD        True       False
4  P000005    Levyjo Buti       Local     Mid    IT        True       False


In [13]:
print("Number of assignment variables:", len(assignments))

Number of assignment variables: 1090187


In [14]:
assigned_rows = []
for (i,j), var in assignments.items():
    if solver.Value(var):
        assigned_rows.append({
            "Person_ID": pers_df.loc[i, "Name_ID"],
            "Person_Name": pers_df.loc[i, "Name"],
            "Pers_Cit": pers_df.loc[i, "Citizenship"],
            "Pers_Tier": pers_df.loc[i, "Tier"],
            "Pers_Field": pers_df.loc[i, "Field"],
            "Pass_1": pers_df.loc[i, "Pass_Test1"],
            "Pass_2": pers_df.loc[i, "Pass_Test2"],
            "Desk_ID": pos_df.loc[j, "Desk_ID"],
            "Req_Cit": pos_df.loc[j, "Req_Citizenship"],
            "Req_Tier": pos_df.loc[j, "Req_Tier"],
            "Req_Field": pos_df.loc[j, "Req_Field"],
            "Desk_Tag": pos_df.loc[j, "Tag"],
            "Tag_Level": pos_df.loc[j, "Level"],
        })


In [15]:
assigned_df = pd.DataFrame(assigned_rows)
assigned_df.to_excel("new_update_assignment_results.xlsx", index=False)